In [1]:
import kagglehub

print(kagglehub.__version__)

1.0.2


In [2]:
from pathlib import Path

# Get the current working directory as an object
dataset_path = Path.cwd() / "movie-review-dataset/"

dataset_path

PosixPath('/content/movie-review-dataset')

In [3]:
dataset_path.mkdir(parents=True, exist_ok=True)

In [4]:
# Download latest version
path = kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews", force_download=True, output_dir=str(dataset_path))

print("Path to dataset files:", path)

100%|██████████| 152M/152M [00:07<00:00, 20.7MB/s]

Extracting files...


Path to dataset files: /content/movie-review-dataset


In [5]:
import shutil

csv_files = {}

for source_files in Path(path).glob("*.csv"):
  if path.startswith("/kaggle/"):
    shutil.copy2(source_files, dataset_path)
  source_file_name = source_files.name
  csv_files[source_file_name] = dataset_path / source_file_name

In [6]:
csv_files

{'rotten_tomatoes_movie_reviews.csv': PosixPath('/content/movie-review-dataset/rotten_tomatoes_movie_reviews.csv'),
 'rotten_tomatoes_movies.csv': PosixPath('/content/movie-review-dataset/rotten_tomatoes_movies.csv')}

In [7]:
!pip install pyspark

## Uso de PySpark

In [8]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as Func

In [9]:
spark = SparkSession.builder.appName("movie-reviews-dataset").getOrCreate()

In [10]:
spark

### Leyendo el dataset CSV en Spark

In [11]:
df_movies = spark.read.csv(
    path=str(csv_files["rotten_tomatoes_movies.csv"]),
    header=True,
    inferSchema=True
)

df_movies.show(3)

+------------------+-------------------+-------------+-----------+------+--------------+-------------------+--------------------+--------------+--------------------+----------------+--------------------+--------------------+---------+-----------+--------+
|                id|              title|audienceScore|tomatoMeter|rating|ratingContents|releaseDateTheaters|releaseDateStreaming|runtimeMinutes|               genre|originalLanguage|            director|              writer|boxOffice|distributor|soundMix|
+------------------+-------------------+-------------+-----------+------+--------------+-------------------+--------------------+--------------+--------------------+----------------+--------------------+--------------------+---------+-----------+--------+
|space-zombie-bingo|Space Zombie Bingo!|           50|       NULL|  NULL|          NULL|               NULL|          2018-08-25|            75|Comedy, Horror, S...|         English|       George Ormrod|George Ormrod,Joh...|     NUL

In [12]:
df_reviews = spark.read.csv(
    path=str(csv_files['rotten_tomatoes_movie_reviews.csv']),
    header=True,
    inferSchema=True
)

df_reviews.show(3)

+--------------------+--------+------------+---------------+-----------+-------------+-----------+--------------------+--------------------+--------------+--------------------+
|                  id|reviewId|creationDate|     criticName|isTopCritic|originalScore|reviewState|      publicatioName|          reviewText|scoreSentiment|           reviewUrl|
+--------------------+--------+------------+---------------+-----------+-------------+-----------+--------------------+--------------------+--------------+--------------------+
|             beavers| 1145982|  2003-05-23|Ivan M. Lincoln|      False|        3.5/4|      fresh|Deseret News (Sal...|Timed to be just ...|      POSITIVE|http://www.desere...|
|          blood_mask| 1636744|  2007-06-02|  The Foywonder|      False|          1/5|     rotten|       Dread Central|It doesn't matter...|      NEGATIVE|http://www.dreadc...|
|city_hunter_shinj...| 2590987|  2019-05-28|   Reuben Baron|      False|         NULL|      fresh|                 

In [13]:
df_movies.dtypes

[('id', 'string'),
 ('title', 'string'),
 ('audienceScore', 'int'),
 ('tomatoMeter', 'int'),
 ('rating', 'string'),
 ('ratingContents', 'string'),
 ('releaseDateTheaters', 'date'),
 ('releaseDateStreaming', 'string'),
 ('runtimeMinutes', 'int'),
 ('genre', 'string'),
 ('originalLanguage', 'string'),
 ('director', 'string'),
 ('writer', 'string'),
 ('boxOffice', 'string'),
 ('distributor', 'string'),
 ('soundMix', 'string')]

In [14]:
# Ver los tipos de columnas, y si son nulas
df_movies.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- audienceScore: integer (nullable = true)
 |-- tomatoMeter: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- ratingContents: string (nullable = true)
 |-- releaseDateTheaters: date (nullable = true)
 |-- releaseDateStreaming: string (nullable = true)
 |-- runtimeMinutes: integer (nullable = true)
 |-- genre: string (nullable = true)
 |-- originalLanguage: string (nullable = true)
 |-- director: string (nullable = true)
 |-- writer: string (nullable = true)
 |-- boxOffice: string (nullable = true)
 |-- distributor: string (nullable = true)
 |-- soundMix: string (nullable = true)



In [15]:
# Cantidad de filas en el dataset
df_movies.count(), df_reviews.count()

(143258, 1446056)

### Limpieza / Selecciones



In [18]:
df_movies.columns

['id',
 'title',
 'audienceScore',
 'tomatoMeter',
 'rating',
 'ratingContents',
 'releaseDateTheaters',
 'releaseDateStreaming',
 'runtimeMinutes',
 'genre',
 'originalLanguage',
 'director',
 'writer',
 'boxOffice',
 'distributor',
 'soundMix']

In [19]:
df_movies_curated = df_movies.select('id', 'title', 'genre', 'audienceScore', 'tomatoMeter', 'releaseDateTheaters', 'releaseDateStreaming')

Quitar / Filtrar las peliculas que tengan al menos una fecha (theaters o streaming)

In [96]:
df_movies_with_date = df_movies_curated.filter(
    Func.col("releaseDateTheaters").isNotNull() |
    Func.col("releaseDateStreaming").isNotNull()
)

In [97]:
df_movies_curated.count(), df_movies_with_date.count()

(143258, 84283)

Conservar solo las peliculas que sean del año >2000

In [98]:
# Primero, escoger la fecha más antigua:

df_movies_2000 = df_movies_with_date.withColumn(
    "releaseDate",
    Func.least(
        Func.try_to_timestamp(Func.col("releaseDateTheaters"), Func.lit('yyyy-MM-dd')),
        Func.try_to_timestamp(Func.col("releaseDateStreaming"), Func.lit('yyyy-MM-dd'))
    )
).filter(Func.year(Func.col("releaseDate")) > 2000)

In [99]:
df_movies_2000.count()

71385

In [100]:
df_movies_2000 = df_movies_2000.withColumn(
    "releaseDate", Func.to_date(Func.col("releaseDate"))
)

Eliminar registros que tengan fechas mal registradas

In [101]:
df_movies_2000.count()

71385

In [102]:
df_movies_2000 = df_movies_2000.filter(
    Func.col("releaseDateTheaters").isNotNull() |
    Func.col("releaseDateStreaming").isNotNull()
)

In [103]:
df_movies_2000.count()

71385

In [28]:
df_movies_2000.dtypes

[('id', 'string'),
 ('title', 'string'),
 ('genre', 'string'),
 ('audienceScore', 'int'),
 ('tomatoMeter', 'int'),
 ('releaseDateTheaters', 'date'),
 ('releaseDateStreaming', 'string'),
 ('releaseDate', 'date')]

In [104]:
df_movies_2000.show(3)

+--------------------+-------------------+--------------------+-------------+-----------+-------------------+--------------------+-----------+
|                  id|              title|               genre|audienceScore|tomatoMeter|releaseDateTheaters|releaseDateStreaming|releaseDate|
+--------------------+-------------------+--------------------+-------------+-----------+-------------------+--------------------+-----------+
|  space-zombie-bingo|Space Zombie Bingo!|Comedy, Horror, S...|           50|       NULL|               NULL|          2018-08-25| 2018-08-25|
|     the_green_grass|    The Green Grass|               Drama|         NULL|       NULL|               NULL|          2020-02-11| 2020-02-11|
|the_sore_losers_1997|        Sore Losers|Action, Mystery &...|           60|       NULL|               NULL|          2020-10-23| 2020-10-23|
+--------------------+-------------------+--------------------+-------------+-----------+-------------------+--------------------+-----------+

Unificando `df_movies_2000` (Peliculas del 2000) con `df_reviews` (Reseñas de peliculas) mediante la columna `id`

In [30]:
df_reviews.columns

['id',
 'reviewId',
 'creationDate',
 'criticName',
 'isTopCritic',
 'originalScore',
 'reviewState',
 'publicatioName',
 'reviewText',
 'scoreSentiment',
 'reviewUrl']

In [105]:
df_movies_reviews_2000 = df_movies_2000.join(df_reviews.select("id", "creationDate", "reviewText", "scoreSentiment", "reviewState"), on=["id"], how="inner")

In [106]:
df_movies_reviews_2000

DataFrame[id: string, title: string, genre: string, audienceScore: int, tomatoMeter: int, releaseDateTheaters: date, releaseDateStreaming: string, releaseDate: date, creationDate: string, reviewText: string, scoreSentiment: string, reviewState: string]

In [107]:
df_movies_reviews_2000.show(3)

+--------------------+--------------------+-------------+-------------+-----------+-------------------+--------------------+-----------+------------+--------------------+--------------+-----------+
|                  id|               title|        genre|audienceScore|tomatoMeter|releaseDateTheaters|releaseDateStreaming|releaseDate|creationDate|          reviewText|scoreSentiment|reviewState|
+--------------------+--------------------+-------------+-------------+-----------+-------------------+--------------------+-----------+------------+--------------------+--------------+-----------+
|             beavers|             Beavers|  Documentary|           75|       NULL|               NULL|          2011-06-21| 2011-06-21|  2003-05-23|Timed to be just ...|      POSITIVE|      fresh|
|small_town_wisconsin|Small Town Wisconsin|Comedy, Drama|           88|         83|         2022-06-03|          2022-06-10| 2022-06-03|  2022-07-22|Small Town Wiscon...|      POSITIVE|      fresh|
|small_tow

Eliminar registros que no contengan el texto de reseñas (`reviewText` nulo o vacio) o algun `scoreSentiment`

In [108]:
df_movies_reviews_2000.count()

1203412

In [109]:
df_movies_reviews_2000 = df_movies_reviews_2000.filter(
    Func.col("reviewText").isNotNull() & (Func.trim(Func.col("reviewText")) != "")
)

In [110]:
df_movies_reviews_2000 = df_movies_reviews_2000.filter(
    Func.col("scoreSentiment").isNotNull()
)

In [111]:
df_movies_reviews_2000.count()

1176576

Revisando la cantidad de caracteres / palabras dentro de las reseñas de peliculas >2010 (min, max, promedio):

In [112]:
df_movies_reviews_2000.schema["reviewText"].dataType

StringType()

In [113]:
df_movies_reviews_2000.select(
    F.min(F.length(F.col("ReviewText"))).alias("min_characters"),
    F.max(F.length(F.col("ReviewText"))).alias("max_characters"),
    F.avg(F.length(F.col("ReviewText"))).alias("avg_characters"),
    F.avg(F.size(F.split(F.col("ReviewText"), " "))).alias("avg_words"),
    F.min(F.size(F.split(F.col("ReviewText"), " "))).alias("min_words"),
    F.max(F.size(F.split(F.col("ReviewText"), " "))).alias("max_words")
).show()

+--------------+--------------+-----------------+------------------+---------+---------+
|min_characters|max_characters|   avg_characters|         avg_words|min_words|max_words|
+--------------+--------------+-----------------+------------------+---------+---------+
|             1|           363|131.3089294699195|21.706636035411226|        1|      153|
+--------------+--------------+-----------------+------------------+---------+---------+



Filtrando reseñas que contengan una cantidad de palabras minima (>30)

In [114]:
df_movies_reviews_2010_clean = df_movies_reviews_2000.filter(
    F.col("reviewText").isNotNull() &
    (F.size(F.split("reviewText", " ")) >= 30)
)

In [115]:
df_movies_reviews_2010_clean.count()

252345

In [116]:
df_movies_reviews_2010_clean.select(
    F.avg(F.size(F.split(F.col("ReviewText"), " "))).alias("avg_words"),
    F.min(F.size(F.split(F.col("ReviewText"), " "))).alias("min_words"),
    F.max(F.size(F.split(F.col("ReviewText"), " "))).alias("max_words")
).show()

+------------------+---------+---------+
|         avg_words|min_words|max_words|
+------------------+---------+---------+
|35.168511363411206|       30|      153|
+------------------+---------+---------+



Problema de caracteres raros / especiales en `reviewText`

Claudo sugiere el uso de `html.unescape`

In [117]:
random_reviews = df_movies_reviews_2010_clean.orderBy(F.rand()).limit(15)

In [118]:
random_reviews.select(
    "reviewText"
).show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 reviewText | The derivative Bratz is a great big pink marshmallow of a movie, aimed at one demographic only: tween girls into fashion and lip gloss. Anyone else, enter at your own risk.                                                                                               
-RECORD 1--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 reviewText | Baz Luhrmann&apos;s loving tribute to  EP&#44; told though the unreliable narration of Hanks as Col&#46; Tom&#46; Austin Butler is a revelat

In [119]:
from pyspark.sql.functions import udf
import html

In [120]:
@udf(returnType=F.StringType())
def remove_characters(reviewText):
  if reviewText is None:
    return None

  return html.unescape(reviewText)

In [121]:
random_reviews.select(
    "reviewText",
    remove_characters("reviewText").alias("clean_reviewText")
).show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 reviewText       | The derivative Bratz is a great big pink marshmallow of a movie, aimed at one demographic only: tween girls into fashion and lip gloss. Anyone else, enter at your own risk.                                                                                               
 clean_reviewText | The derivative Bratz is a great big pink marshmallow of a movie, aimed at one demographic only: tween girls into fashion and lip gloss. Anyone else, enter at your own risk.                                                                                               
-RECORD 1-------------------------------------------------------------------------------------------------------------------------------

## Misc

In [57]:
import pyspark.sql.functions as F

In [55]:
df_movies_reviews_2000.select(
    "reviewText",
    F.length("reviewText").alias("len"),
    F.upper("reviewText").alias("uppercase"),
    F.substring("reviewText", 1, 5).alias("sliced")
).show(5, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 reviewText | Timed to be just long enough for most youngsters' brief attention spans -- and it's packed with plenty of interesting activity, both on land and under the water.                                     
 len        | 161                                                                                                                                                                                                   
 uppercase  | TIMED TO BE JUST LONG ENOUGH FOR MOST YOUNGSTERS' BRIEF ATTENTION SPANS -- AND IT'S PACKED WITH PLENTY OF INTERESTING ACTIVITY, BOTH ON LAND AND UNDER THE WATER.                                     
 sliced     | Timed                                                                                                                                 